# nb_validacao_dados — Auditoria de Qualidade Censo & Mercado (v5.2 Code-Join)

Esta versão realiza o JOIN pelos códigos IBGE (7 dígitos) para garantir 100% de precisão no cruzamento.

In [ ]:
import pyodbc
import pandas as pd

SERVER   = 'ena6obg6j2cevcppw7dn7yu57a-knnp5frchjbujdik4l3nmgrgsa.datawarehouse.fabric.microsoft.com'
DATABASE = 'lh_dados_publicos'

def get_sql_conn():
    return pyodbc.connect(
        f'Driver={{ODBC Driver 18 for SQL Server}};'
        f'Server={SERVER};'
        f'Database={DATABASE};'
        'Encrypt=yes;'
        'TrustServerCertificate=no;'
        'Authentication=ActiveDirectoryInteractive'
    )

def sql_query(query_str):
    with get_sql_conn() as conn:
        return pd.read_sql(query_str, conn)

def discover_columns(table_name):
    try:
        df = sql_query(f"SELECT TOP 0 * FROM {table_name}")
        return list(df.columns)
    except:
        return []

print('[OK] Conexão preparada via SQL Endpoint.')

In [ ]:
tabelas = [
    'gold_censo_piramide_populacao',
    'gold_censo_genero',
    'gold_censo_renda',
    'gold_censo_fecundidade',
    'gold_censo_envelhecimento',
    'gold_censo_dependencia_demografica',
    'gold_mercado_trabalho'
]

print("--- Scanner de Metadados Completo ---")
meta = {}
for t in tabelas:
    cols = discover_columns(t)
    meta[t] = cols
    status = f"✅ {len(cols)} colunas" if cols else "❌ Tabela não visível"
    print(f"{t.ljust(35)}: {status}")

In [ ]:
print("--- Cruzamento Censo x Mercado (JOIN por Código) ---")

cols_mt = meta.get('gold_mercado_trabalho', [])
col_valor_mt = next((c for c in cols_mt if c.lower() in ['vinculos_ativos', 'valor', 'quantidade_vinculos_ativos']), 'valor')
col_mun_mt = 'nome_municipio'
col_id_mt = 'id_municipio'

query_cruz = f"""
    WITH CensoPop AS (
        SELECT 
            municipio, 
            municipio_codigo, 
            SUM(TRY_CAST(REPLACE(valor, ',', '.') AS FLOAT)) as pop_total
        FROM gold_censo_piramide_populacao
        WHERE ano = '2022'
        GROUP BY municipio, municipio_codigo
    ),
    MercadoTrabalho AS (
        SELECT 
            {col_id_mt} as id_mun, 
            {col_mun_mt} as nome_mun, 
            SUM(TRY_CAST({col_valor_mt} AS FLOAT)) as empregos
        FROM gold_mercado_trabalho
        WHERE ano = 2022
        GROUP BY {col_id_mt}, {col_mun_mt}
    )
    SELECT 
        c.municipio, 
        c.pop_total, 
        m.empregos,
        (m.empregos / NULLIF(c.pop_total, 0)) * 100 as percentual_formalizacao
    FROM CensoPop c
    -- JOIN POR CÓDIGO IBGE (Muito mais seguro que nome)
    INNER JOIN MercadoTrabalho m ON c.municipio_codigo = m.id_mun
    ORDER BY percentual_formalizacao DESC
"""

try:
    res = sql_query(query_cruz)
    if res.empty:
        print("⚠️ Cruzamento retornou vazio. Verifique se os códigos de municípios coincidem.")
    else:
        print(res.to_string(index=False))
except Exception as e:
    print(f"❌ Falha no cruzamento: {e}")


# Análise Exploratória — Gráficos Python

Validação visual das Gold tables antes do modelo semântico.  
Cada gráfico lê direto do SQL Endpoint via `sql_query()` já configurado acima.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

CLUSTER_CORES = {
    'Santos': '#1f77b4',
    'Osasco': '#ff7f0e',
    'Mauá':   '#2ca02c',
}

def cor_cluster(cluster_series):
    return cluster_series.map(CLUSTER_CORES).fillna('#999999')

print('[OK] matplotlib configurado · clusters: Santos=azul · Osasco=laranja · Mauá=verde')

## Gráfico 1 — % Formalização por Município (2022)

Cruzamento `gold_mercado_trabalho` (vínculos RAIS) ÷ `gold_populacao_municipios`.  
Permite ver quais municípios têm mercado formal mais robusto relativo à população.

In [ ]:
df_formal = sql_query("""
    WITH rais AS (
        SELECT nome_municipio, cluster,
               SUM(CAST(vinculos_ativos AS FLOAT)) AS vinculos
        FROM gold_mercado_trabalho
        WHERE fonte = 'RAIS' AND ano = 2022
        GROUP BY nome_municipio, cluster
    ),
    pop AS (
        SELECT nome_municipio,
               SUM(TRY_CAST(valor AS FLOAT)) AS populacao
        FROM gold_populacao_municipios
        WHERE ano = 2022
        GROUP BY nome_municipio
    )
    SELECT r.nome_municipio, r.cluster,
           r.vinculos, p.populacao,
           ROUND((r.vinculos / NULLIF(p.populacao, 0)) * 100, 1) AS pct_formalizacao
    FROM rais r
    JOIN pop p ON r.nome_municipio = p.nome_municipio
    ORDER BY pct_formalizacao DESC
""")

print(df_formal.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
cores = cor_cluster(df_formal['cluster'])
bars = ax.barh(df_formal['nome_municipio'], df_formal['pct_formalizacao'], color=cores)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_xlabel('% Formalização (vínculos RAIS / população)')
ax.set_title('% Formalização por Município — 2022', fontweight='bold')
ax.invert_yaxis()

legend = [mpatches.Patch(color=c, label=k) for k, c in CLUSTER_CORES.items()]
ax.legend(handles=legend, loc='lower right')
plt.tight_layout()
plt.show()

## Gráfico 2 — Saldo CAGED por Cluster (2020–2025)

Evolução do fluxo de emprego formal (admissões − desligamentos) agregado por cluster.  
Valida se o `gold_mercado_trabalho` tem cobertura temporal consistente.

In [ ]:
df_caged = sql_query("""
    SELECT cluster, ano,
           SUM(CAST(saldo_mensal AS FLOAT)) AS saldo_anual
    FROM gold_mercado_trabalho
    WHERE fonte = 'CAGED'
      AND ano BETWEEN 2020 AND 2025
    GROUP BY cluster, ano
    ORDER BY cluster, ano
""")

print(df_caged.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 5))
for cluster, grp in df_caged.groupby('cluster'):
    cor = CLUSTER_CORES.get(cluster, '#999')
    ax.plot(grp['ano'], grp['saldo_anual'], marker='o', label=cluster, color=cor, linewidth=2)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Ano')
ax.set_ylabel('Saldo líquido de empregos')
ax.set_title('Saldo CAGED Anual por Cluster — 2020–2025', fontweight='bold')
ax.legend()
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

## Gráfico 3 — Índice de Envelhecimento por Município

Razão entre população idosa (65+) e jovem (0–14) × 100.  
Valores > 100 indicam mais idosos que jovens — municípios em transição demográfica avançada.

In [ ]:
df_env = sql_query("""
    SELECT nome_municipio, cluster,
           TRY_CAST(indice_envelhecimento_2022 AS FLOAT) AS indice_2022,
           TRY_CAST(indice_envelhecimento_2010 AS FLOAT) AS indice_2010
    FROM gold_censo_envelhecimento
    ORDER BY indice_2022 DESC
""")

print(df_env.to_string(index=False))

x = np.arange(len(df_env))
w = 0.35
cores = cor_cluster(df_env['cluster'])

fig, ax = plt.subplots(figsize=(12, 5))
bars2022 = ax.bar(x - w/2, df_env['indice_2022'], w, label='2022', color=cores, alpha=0.9)
bars2010 = ax.bar(x + w/2, df_env['indice_2010'], w, label='2010', color=cores, alpha=0.45)

ax.axhline(100, color='red', linewidth=0.9, linestyle='--', label='Equilíbrio (100)')
ax.set_xticks(x)
ax.set_xticklabels(df_env['nome_municipio'], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Índice de Envelhecimento')
ax.set_title('Índice de Envelhecimento por Município — 2010 vs 2022', fontweight='bold')

legend_clusters = [mpatches.Patch(color=c, label=k) for k, c in CLUSTER_CORES.items()]
ax.legend(handles=legend_clusters + [
    mpatches.Patch(color='gray', alpha=0.9, label='2022'),
    mpatches.Patch(color='gray', alpha=0.45, label='2010'),
])
plt.tight_layout()
plt.show()

## Gráfico 4 — Renda Domiciliar per Capita: 2010 vs 2022

Comparativo de renda por município nos dois censos.  
Revela quais municípios mais avançaram (ou regrediram) em poder aquisitivo domiciliar.

In [ ]:
df_renda = sql_query("""
    SELECT nome_municipio, cluster,
           TRY_CAST(renda_per_capita_2010 AS FLOAT) AS renda_2010,
           TRY_CAST(renda_per_capita_2022 AS FLOAT) AS renda_2022
    FROM gold_censo_renda
    ORDER BY renda_2022 DESC
""")

print(df_renda.to_string(index=False))

x = np.arange(len(df_renda))
w = 0.35
cores = cor_cluster(df_renda['cluster'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w/2, df_renda['renda_2010'], w, color=cores, alpha=0.45, label='2010')
ax.bar(x + w/2, df_renda['renda_2022'], w, color=cores, alpha=0.95, label='2022')

ax.set_xticks(x)
ax.set_xticklabels(df_renda['nome_municipio'], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Renda domiciliar per capita (R$)')
ax.set_title('Renda Domiciliar per Capita por Município — 2010 vs 2022', fontweight='bold')

legend_clusters = [mpatches.Patch(color=c, label=k) for k, c in CLUSTER_CORES.items()]
ax.legend(handles=legend_clusters + [
    mpatches.Patch(color='gray', alpha=0.45, label='2010'),
    mpatches.Patch(color='gray', alpha=0.95, label='2022'),
])
plt.tight_layout()
plt.show()

## Gráfico 5 — Pirâmide Etária (município selecionável)

Visualiza a estrutura etária por sexo.  
Altere `MUNICIPIO_ALVO` para comparar qualquer um dos 15 municípios.

In [ ]:
MUNICIPIO_ALVO = 'Santos'  # <-- altere aqui

df_pir = sql_query(f"""
    SELECT faixa_etaria, sexo,
           SUM(TRY_CAST(valor AS FLOAT)) AS populacao
    FROM gold_censo_piramide_populacao
    WHERE nome_municipio = '{MUNICIPIO_ALVO}'
      AND ano = '2022'
    GROUP BY faixa_etaria, sexo
    ORDER BY faixa_etaria
""")

print(df_pir.to_string(index=False))

homens  = df_pir[df_pir['sexo'].str.lower().str.contains('homem|masculin', na=False)]
mulheres = df_pir[df_pir['sexo'].str.lower().str.contains('mulher|feminin', na=False)]

faixas = homens['faixa_etaria'].tolist()
pop_h  = homens['populacao'].tolist()
pop_m  = mulheres['populacao'].tolist()

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(faixas, [-v for v in pop_h], color='#4c72b0', label='Homens')
ax.barh(faixas, pop_m, color='#dd8452', label='Mulheres')

max_val = max(pop_h + pop_m)
ticks = np.linspace(0, max_val, 5).astype(int)
ax.set_xticks([-t for t in ticks] + list(ticks))
ax.set_xticklabels([f'{abs(t):,}' for t in ticks] + [f'{t:,}' for t in ticks], fontsize=8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Pirâmide Etária — {MUNICIPIO_ALVO} (Censo 2022)', fontweight='bold')
ax.set_xlabel('População')
ax.legend()
plt.tight_layout()
plt.show()

## Gráfico 6 — Top 5 Setores CNAE por Cluster (vínculos RAIS 2022)

Identifica a especialização econômica de cada cluster.  
Útil para validar se o `gold_mercado_trabalho` está capturando a diversidade setorial corretamente.

In [ ]:
df_cnae = sql_query("""
    SELECT cluster, secao_cnae,
           SUM(CAST(vinculos_ativos AS FLOAT)) AS vinculos
    FROM gold_mercado_trabalho
    WHERE fonte = 'RAIS' AND ano = 2022
      AND secao_cnae IS NOT NULL
    GROUP BY cluster, secao_cnae
""")

clusters_ord = ['Santos', 'Osasco', 'Mauá']
fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=False)

for ax, cluster in zip(axes, clusters_ord):
    grp = (df_cnae[df_cnae['cluster'] == cluster]
           .nlargest(5, 'vinculos')
           .sort_values('vinculos'))
    cor = CLUSTER_CORES.get(cluster, '#999')
    bars = ax.barh(grp['secao_cnae'], grp['vinculos'], color=cor, alpha=0.85)
    ax.bar_label(bars, fmt='{:,.0f}', padding=3, fontsize=8)
    ax.set_title(f'Cluster {cluster}', fontweight='bold', color=cor)
    ax.set_xlabel('Vínculos ativos')
    ax.tick_params(axis='y', labelsize=8)

fig.suptitle('Top 5 Seções CNAE por Cluster — RAIS 2022', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()